# Mind2Web Sanity Check

Verify the real Multimodal-Mind2Web data pipeline:
1. Dataset loading
2. Single-sample field inspection
3. BBox / click-point visualization
4. Reward computation demo
5. Metrics demo

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))
print(f"Project root: {project_root}")

## 1. Load real Mind2Web data

In [ ]:
from gui_grounding.data.mind2web_dataset import Mind2WebDataset

ds = Mind2WebDataset(split="train", max_samples=10, cache_screenshots=True)
print(f"Loaded {len(ds)} samples")

## 2. Inspect a single sample

In [ ]:
sample = ds[0]

print(f"sample_id:          {sample.sample_id}")
print(f"instruction:        {sample.instruction}")
print(f"action_type:        {sample.action_type}")
print(f"target_element_id:  {sample.target_element_id}")
print(f"target_bbox:        {sample.target_bbox.as_tuple() if sample.target_bbox else None}")
print(f"click_point:        {sample.click_point}")
print(f"image_path:         {sample.image_path}")
print(f"website:            {sample.website}")
print(f"domain:             {sample.domain}")
print(f"platform:           {sample.platform}")
print(f"dom_candidates:     {len(sample.dom_candidates) if sample.dom_candidates else 0}")
print()
print("Metadata:")
for k, v in sample.metadata.items():
    print(f"  {k}: {v}")

## 3. Visualize bbox & click point on screenshot

In [ ]:
from gui_grounding.utils.visualization import draw_prediction

img = ds.get_screenshot(sample)
if img is not None:
    gt_bbox = sample.target_bbox.as_tuple() if sample.target_bbox else None
    gt_point = sample.click_point

    annotated = draw_prediction(
        img,
        gt_bbox=gt_bbox,
        gt_point=gt_point,
        action_type=sample.action_type,
    )

    # Crop to the region of interest for display
    if gt_bbox:
        x1, y1, x2, y2 = gt_bbox
        margin = 150
        crop = annotated.crop((
            max(0, x1 - margin),
            max(0, y1 - margin),
            min(img.width, x2 + margin),
            min(img.height, y2 + margin),
        ))
        display(crop)
    else:
        # Show top portion
        display(annotated.crop((0, 0, img.width, min(800, img.height))))
else:
    print("Screenshot not available")

## 4. Reward computation demo

In [ ]:
from gui_grounding.reward.verifiable_reward import VerifiableRewardCalculator
from gui_grounding.reward.candidate_generator import CandidateGenerator

calc = VerifiableRewardCalculator()
gen = CandidateGenerator(mode="dummy", num_candidates=8, seed=42)

candidates = gen.generate(sample)

gt_bbox = sample.target_bbox.as_tuple() if sample.target_bbox else None
print(f"Sample: {sample.sample_id}")
print(f"GT bbox: {gt_bbox}")
print(f"GT action: {sample.action_type}")
print()

for c in candidates:
    result = calc.compute(
        sample_id=c.candidate_id,
        pred_bbox=c.bbox.as_tuple() if c.bbox else None,
        gt_bbox=gt_bbox,
        pred_click=c.click_point,
        pred_action=c.action_type,
        gt_action=sample.action_type,
        pred_element_id=c.element_id,
        gt_element_id=sample.target_element_id,
    )
    print(f"  {c.candidate_id}: reward={result.total_reward:.4f}  "
          f"(iou={result.components.iou:.3f}, click={result.components.click_inside_target:.0f}, "
          f"act={result.components.action_type_correct:.0f})")

## 5. Metrics demo

In [ ]:
from gui_grounding.evaluation.metrics import compute_all_metrics

# Use first candidate as dummy prediction for each sample
pred_eids, gt_eids = [], []
pred_bboxes, gt_bboxes = [], []
pred_pts, pred_acts, gt_acts = [], [], []

for s in ds:
    cands = gen.generate(s)
    c0 = cands[0]
    gt_eids.append(s.target_element_id)
    gt_bboxes.append(s.target_bbox.as_tuple() if s.target_bbox else None)
    gt_acts.append(s.action_type)
    pred_eids.append(c0.element_id)
    pred_bboxes.append(c0.bbox.as_tuple() if c0.bbox else None)
    pred_pts.append(c0.click_point)
    pred_acts.append(c0.action_type)

metrics = compute_all_metrics(
    pred_element_ids=pred_eids, gt_element_ids=gt_eids,
    pred_bboxes=pred_bboxes, gt_bboxes=gt_bboxes,
    pred_points=pred_pts,
    pred_actions=pred_acts, gt_actions=gt_acts,
)

print("Metrics (dummy candidates — NOT real model predictions):")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")